# 05 - Visualization

Visualize trained models: potential slices, radial curves, training metrics, and DF marginals.

This notebook demonstrates how to call all the plotting/eval utilities from Jupyter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dpjax.paths import PROJECT_ROOT, RUNS_DIR, DATA_DIR

# Paths – adjust if your run dirs differ
DATA_PATH    = DATA_DIR / "plummer_n131072.h5"
DF_RUN_DIR   = RUNS_DIR / "plummer" / "df"
PHI_RUN_DIR  = RUNS_DIR / "plummer" / "phi"

## 1. Training Curves

Plot loss and diagnostic statistics from `metrics.csv`.

In [ ]:
import csv

def read_metrics(run_dir):
    path = run_dir / "metrics.csv"
    if not path.exists():
        raise FileNotFoundError(f"No metrics.csv in {run_dir}")
    with path.open() as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    return {k: np.array([float(r[k]) for r in rows]) for k in rows[0].keys()}

# DF metrics
try:
    df_m = read_metrics(DF_RUN_DIR)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(df_m["step"], df_m["loss"], lw=1.2)
    ax1.set_xlabel("step"); ax1.set_ylabel("NLL"); ax1.set_title("DF Loss")
    ax1.grid(True, alpha=0.2)
    ax2.plot(df_m["step"], df_m["score_p99"], lw=1.2, color="tab:orange")
    ax2.set_xlabel("step"); ax2.set_ylabel("|score| p99"); ax2.set_title("Score p99")
    ax2.grid(True, alpha=0.2)
    fig.suptitle("DF Training", fontsize=13); fig.tight_layout(); plt.show()
except FileNotFoundError as e:
    print(e)

In [ ]:
# Phi metrics
try:
    phi_m = read_metrics(PHI_RUN_DIR)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(phi_m["step"], phi_m["loss"], lw=1.2)
    axes[0].set_xlabel("step"); axes[0].set_ylabel("loss"); axes[0].set_title("Phi Loss")
    axes[0].grid(True, alpha=0.2)
    axes[1].plot(phi_m["step"], phi_m["residual_std"], lw=1.2, color="tab:orange")
    axes[1].set_xlabel("step"); axes[1].set_ylabel("std"); axes[1].set_title("Residual Std")
    axes[1].grid(True, alpha=0.2)
    axes[2].plot(phi_m["step"], phi_m["residual_p99_abs"], lw=1.2, color="tab:red")
    axes[2].set_xlabel("step"); axes[2].set_ylabel("|r| p99"); axes[2].set_title("Residual p99")
    axes[2].grid(True, alpha=0.2)
    fig.suptitle("Phi Training", fontsize=13); fig.tight_layout(); plt.show()
except FileNotFoundError as e:
    print(e)

## 2. Evaluate Phi: Radial Curves

In [ ]:
from experiments.eval_phi import run_eval_phi

try:
    res = run_eval_phi(
        data_path=DATA_PATH,
        df_run_dir=DF_RUN_DIR,
        phi_run_dir=PHI_RUN_DIR,
        n_eval=16384,
    )

    rad = res["radial"]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    ax1.plot(rad["r"], rad["phi_true"], "k-", lw=2, label="Plummer analytic")
    ax1.plot(rad["r"], rad["phi_learned_shift"], "--", lw=1.5, label="Learned")
    ax1.set_xscale("log"); ax1.set_xlabel("r"); ax1.set_ylabel(r"$\Phi(r)$")
    ax1.legend(); ax1.set_title("Potential"); ax1.grid(True, alpha=0.2)

    ax2.plot(rad["r"], rad["ar_true"], "k-", lw=2, label="Plummer analytic")
    ax2.plot(rad["r"], rad["ar_learned"], "--", lw=1.5, label="Learned")
    ax2.set_xscale("log"); ax2.set_xlabel("r"); ax2.set_ylabel(r"$a_r(r)$")
    ax2.legend(); ax2.set_title("Radial Acceleration"); ax2.grid(True, alpha=0.2)

    fig.tight_layout(); plt.show()

    print("\nResidual stats:")
    for k, v in res["stats"].items():
        print(f"  {k}: {v}")
except Exception as e:
    print(f"Skipped: {e}")

## 3. Evaluate DF: Marginal Comparisons

In [ ]:
from experiments.eval_df import run_eval_df

try:
    df_res = run_eval_df(
        data_path=DATA_PATH,
        df_run_dir=DF_RUN_DIR,
        n_samples=50000,
    )
    print(f"DF evaluation plots saved to: {df_res['out_dir']}")
except Exception as e:
    print(f"Skipped: {e}")

## 4. Potential Slice Visualization

Render $\Phi(x,y)$, $\rho(x,y)$, and $|a(x,y)|$ at a fixed $z$ slice.

In [ ]:
import jax
import jax.numpy as jnp
import yaml
from dpjax.data import Normalizer
from dpjax.models.potential import PotentialConfig, PotentialMLP
from dpjax.utils.ckpt import create_manager, restore_latest

try:
    # Load normalizer and phi model
    normalizer = Normalizer.load_npz(DF_RUN_DIR / "normalizer.npz")
    phi_cfg = yaml.safe_load((PHI_RUN_DIR / "config.yaml").read_text())
    pot_cfg = phi_cfg.get("potential", {})
    phi_model = PotentialMLP(PotentialConfig(
        hidden_sizes=tuple(int(x) for x in pot_cfg.get("hidden_sizes", [256, 256, 256]))
    ))
    ckpt_mgr = create_manager(PHI_RUN_DIR / "ckpt")
    phi_params = restore_latest(ckpt_mgr)["params"]

    mean_x = np.asarray(normalizer.mean[:3], dtype=np.float32)
    std_x = np.asarray(normalizer.std[:3], dtype=np.float32)

    # Grid
    rmax, grid = 5.0, 100
    xs = np.linspace(-rmax, rmax, grid, dtype=np.float32)
    X, Y = np.meshgrid(xs, xs, indexing="xy")
    xyz = np.stack([X.ravel(), Y.ravel(), np.zeros(X.size, dtype=np.float32)], axis=-1)
    xyz_std = (xyz - mean_x[None, :]) / std_x[None, :]

    def phi_single(xi):
        return phi_model.apply({"params": phi_params}, xi)

    @jax.jit
    def eval_batch(x_std_b):
        phi_b = jax.vmap(phi_single)(x_std_b)
        hess_b = jax.vmap(jax.hessian(phi_single))(x_std_b)
        diag = jnp.stack([hess_b[:, 0, 0], hess_b[:, 1, 1], hess_b[:, 2, 2]], axis=-1)
        lap = jnp.sum(diag / (jnp.asarray(std_x)**2)[None, :], axis=-1)
        rho_b = lap / (4.0 * jnp.pi)
        return phi_b, rho_b

    phi_vals, rho_vals = [], []
    bs = 2048
    for i in range(0, xyz_std.shape[0], bs):
        p, rh = eval_batch(jnp.asarray(xyz_std[i:i+bs]))
        phi_vals.append(np.asarray(p)); rho_vals.append(np.asarray(rh))

    phi_img = np.concatenate(phi_vals).reshape(X.shape)
    rho_img = np.concatenate(rho_vals).reshape(X.shape)

    from matplotlib import colors

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

    phi0 = phi_img - np.nanmean(phi_img)
    vmin, vmax = np.nanpercentile(phi0, [1, 99])
    im1 = ax1.imshow(phi0, extent=[-rmax, rmax, -rmax, rmax], origin="lower",
                     cmap="seismic", norm=colors.TwoSlopeNorm(vcenter=0, vmin=vmin, vmax=vmax))
    ax1.set_xlabel("x"); ax1.set_ylabel("y")
    ax1.set_title(r"$\Phi(x,y,0)$ (mean-subtracted)")
    fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

    rho_pos = np.clip(rho_img, 1e-12, np.inf)
    im2 = ax2.imshow(rho_pos, extent=[-rmax, rmax, -rmax, rmax], origin="lower",
                     cmap="magma", norm=colors.LogNorm(
                         vmin=np.nanpercentile(rho_pos, 5), vmax=np.nanpercentile(rho_pos, 99)))
    ax2.set_xlabel("x"); ax2.set_ylabel("y")
    ax2.set_title(r"$\rho(x,y,0) = \nabla^2\Phi / 4\pi$")
    fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

    fig.tight_layout(); plt.show()
except Exception as e:
    print(f"Skipped: {e}")